In [ ]:
"""CareLoop Clinics — Part 1: ingest, validate, clean and enrich.
Run from the project root. Original CSVs in data/ are never modified.
"""
from pathlib import Path
import csv, json, time, os
import pandas as pd
import numpy as np
import requests

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

APPOINTMENTS = DATA / "appointments.csv"
CLINICS = DATA / "clinics.csv"

# Task 1.1 — csv module only (not pandas)
with APPOINTMENTS.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    raw_rows = list(reader)
    headers = reader.fieldnames
print("Task 1.1 — headers:", headers)
print("Task 1.1 — rows:", len(raw_rows))

def parse_row(row):
    """Convert the four requested numeric fields; reject conversion failures."""
    converted = dict(row)
    try:
        converted["age"] = int(row["age"])
        converted["fee_inr"] = int(row["fee_inr"])
        converted["prior_no_shows"] = int(row["prior_no_shows"])
        converted["distance_km"] = float(row["distance_km"])
        return converted
    except (ValueError, TypeError, KeyError):
        return None

accepted, rejected = [], []
for row in raw_rows:
    parsed = parse_row(row)
    if parsed is None:
        rejected.append(row)
    else:
        accepted.append(parsed)

with (OUT / "rejected_rows.json").open("w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2, ensure_ascii=False)
print(f"Task 1.1 — accepted: {len(accepted)}; rejected: {len(rejected)}")

# Task 1.2 — pandas inspection and first cleaning pass
raw = pd.read_csv(APPOINTMENTS)
print("Task 1.2 — raw shape:", raw.shape)
print("\ninfo/nulls/duplicates:")
raw.info()
print(raw.isnull().sum())
print("Exact duplicate rows:", raw.duplicated().sum())

clean = raw.drop_duplicates().copy()
clean["booking_channel"] = (
    clean["booking_channel"].astype("string").str.strip().str.title()
)
clean["appt_date"] = pd.to_datetime(clean["appt_date"], errors="coerce")
clean["booked_on"] = pd.to_datetime(clean["booked_on"], errors="coerce")
unparseable_appt_dates = int(clean["appt_date"].isna().sum())
clean = clean.dropna(subset=["appt_date"]).copy()

# The target is status-derived. Rows with no status cannot be labelled for
# supervised ML or counted as attended/no-show, so exclude them transparently.
missing_status = int(clean["status"].isna().sum())
clean = clean.dropna(subset=["status"]).copy()
print("Rows excluded for unparseable appt_date:", unparseable_appt_dates)
print("Rows excluded for missing status after date cleaning:", missing_status)

# Task 1.3 — validity, missing values and IQR capping
invalid_age = (clean["age"] < 0) | (clean["age"] > 100)
invalid_age_count = int(invalid_age.sum())
clean.loc[invalid_age, "age"] = np.nan
age_median = clean["age"].median()
distance_median = clean["distance_km"].median()
clean["age"] = clean["age"].fillna(round(age_median))
clean["distance_km"] = clean["distance_km"].fillna(distance_median)

cap_counts = {}
fences = {}
for column in ["distance_km", "fee_inr"]:
    q1 = clean[column].quantile(0.25)
    q3 = clean[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    cap_counts[column] = int(((clean[column] < lower) | (clean[column] > upper)).sum())
    fences[column] = {"q1": float(q1), "q3": float(q3), "lower": float(lower), "upper": float(upper)}
    clean[column] = clean[column].clip(lower, upper)

clean["distance_km"] = clean["distance_km"].round(1)
clean["fee_inr"] = clean["fee_inr"].round().astype(int)
clean["age"] = clean["age"].round().astype(int)

# Task 1.4 — features and NumPy summary
clean["lead_days"] = (clean["appt_date"] - clean["booked_on"]).dt.days
clean["no_show"] = (clean["status"] == "No-Show").astype(int)
X_numeric = clean[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
lead_mask = clean["lead_days"] > 14
z_distance = (clean["distance_km"] - np.mean(clean["distance_km"])) / np.std(clean["distance_km"], ddof=0)
print("Task 1.4 — numeric means:", np.round(X_numeric.mean(axis=0), 2))
print("lead_days > 14:", int(lead_mask.sum()), "share:", round(float(lead_mask.mean()), 4))
print("distance |z| > 2:", int((np.abs(z_distance) > 2).sum()))

# Task 1.5 — Open-Meteo first hourly value (midnight UTC), no API key
clinics = pd.read_csv(CLINICS)
def fetch_midnight_utc_temperature(row):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }
    last_error = None
    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=15)
            response.raise_for_status()
            payload = response.json()
            values = payload.get("hourly", {}).get("temperature_2m", [])
            if not values:
                raise ValueError("hourly.temperature_2m missing or empty")
            return float(values[0])
        except Exception as exc:
            last_error = exc
            if attempt == 0:
                time.sleep(1)
    print(f"API failed for {row['clinic_name']}: {last_error}; temp_c=NULL")
    return np.nan

if os.getenv("CARELOOP_SKIP_WEATHER") == "1":
    # Offline build mode: keep the column, then rerun without this flag to fetch live values.
    clinics["temp_c"] = np.nan
    print("Weather API skipped in offline build mode; temp_c is NULL until rerun online.")
else:
    clinics["temp_c"] = clinics.apply(fetch_midnight_utc_temperature, axis=1)
clinics.to_csv(OUT / "clinics_enriched.csv", index=False)

# Task 1.6 — exact 16-column output, dates as YYYY-MM-DD text
columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"
]
clean = clean.sort_values("appointment_id").copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")
clean = clean[columns]
assert clean["appointment_id"].is_unique
assert clean[columns].notna().all().all()
clean.to_csv(OUT / "appointments_clean.csv", index=False)

# SQL INSERT script: clinics first (foreign key parent), then appointments
# Escape quotes and emit NULL for missing clinic temperatures.
def sql_value(value):
    if pd.isna(value):
        return "NULL"
    if isinstance(value, (int, np.integer, float, np.floating)):
        return str(value)
    return "'" + str(value).replace("'", "''") + "'"

with (OUT / "load_data.sql").open("w", encoding="utf-8") as f:
    f.write("USE noshowguard;\n\n")
    for _, row in clinics.iterrows():
        vals = [row["clinic_id"], row["clinic_name"], row["city"], row["tier"], row["latitude"], row["longitude"], row["temp_c"]]
        f.write("INSERT INTO clinics VALUES (" + ", ".join(sql_value(v) for v in vals) + ");\n")
    f.write("\n")
    for _, row in clean.iterrows():
        f.write("INSERT INTO appointments VALUES (" + ", ".join(sql_value(v) for v in row.tolist()) + ");\n")

summary = {
    "source_rows": int(len(raw)),
    "csv_module_accepted": int(len(accepted)),
    "csv_module_rejected": int(len(rejected)),
    "exact_duplicates_removed": int(raw.duplicated().sum()),
    "unparseable_appt_dates_removed_after_deduplication": unparseable_appt_dates,
    "invalid_age_values_changed_to_missing": invalid_age_count,
    "age_median_used": float(age_median),
    "distance_median_used": float(distance_median),
    "iqr_fences": fences,
    "values_capped": cap_counts,
    "missing_status_rows_excluded_for_target": missing_status,
    "clean_rows": int(len(clean)),
    "clean_columns": int(len(clean.columns)),
    "no_shows": int(clean["no_show"].sum()),
    "no_show_rate_pct": round(float(clean["no_show"].mean() * 100), 1),
    "lead_days_min": int(clean["lead_days"].min()),
    "lead_days_max": int(clean["lead_days"].max()),
    "lead_days_gt_14_count": int(lead_mask.sum()),
    "distance_abs_z_gt_2_count": int((np.abs(z_distance) > 2).sum()),
    "clinic_temperature_missing_count": int(clinics["temp_c"].isna().sum()),
}
with (OUT / "part1_audit.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("\nPART 1 COMPLETE")
print(json.dumps(summary, indent=2))
print("Saved outputs:", sorted(p.name for p in OUT.iterdir()))


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/appointments.csv'

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving appointments.csv to appointments (1).csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving clinics.csv to clinics (1).csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving followup_notes.csv to followup_notes (1).csv


In [ ]:
import os

os.makedirs("/content/CareLoop/data", exist_ok=True)
os.makedirs("/content/CareLoop/outputs", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [ ]:
import os
import shutil

for filename in ["appointments.csv", "clinics.csv", "followup_notes.csv"]:
    source = "/content/" + filename
    destination = "/content/CareLoop/data/" + filename

    if os.path.exists(source):
        shutil.move(source, destination)
        print(filename, "moved successfully!")
    elif os.path.exists(destination):
        print(filename, "already moved!")
    else:
        print(filename, "NOT FOUND")

appointments.csv moved successfully!
clinics.csv moved successfully!
followup_notes.csv moved successfully!


In [ ]:
import os

print("Files inside CareLoop/data:")

for file in os.listdir("/content/CareLoop/data"):
    print(file)

Files inside CareLoop/data:
clinics.csv
followup_notes.csv
appointments.csv


In [ ]:
import csv

APPOINTMENTS = "/content/CareLoop/data/appointments.csv"

with open(APPOINTMENTS, "r", encoding="cp1252", newline="") as f:
    reader = csv.DictReader(f)
    raw_rows = list(reader)
    headers = reader.fieldnames

print("Task 1.1 - Headers:", headers)
print("Task 1.1 - Total rows:", len(raw_rows))

Task 1.1 - Headers: ['bplist00Ñ\x01\x02_\x10\x0fWebMainResourceÕ\x03\x04\x05\x06\x07\x08\t']
Task 1.1 - Total rows: 1023


In [ ]:
import os

file_path = "/content/CareLoop/data/appointments.csv"

with open(file_path, "rb") as f:
    print(f.read(100))


b'bplist00\xd1\x01\x02_\x10\x0fWebMainResource\xd5\x03\x04\x05\x06\x07\x08\t\n\x0b\x0c^WebResourceURL_\x10\x1bWebResourceTextEncodingName_\x10\x0fWebResourceD'


In [ ]:
import os

folder = "/content/CareLoop/data"

for filename in os.listdir(folder):
    path = os.path.join(folder, filename)
    with open(path, "rb") as f:
        print(filename, ":", f.read(8))

clinics.csv : b'bplist00'
followup_notes.csv : b'bplist00'
appointments.csv : b'bplist00'
